In [1]:
import ee

In [15]:
ee.Authenticate()

True

In [14]:
ee.Initialize(project='satexport')

In [16]:


# --- AOI: quadrado ~30 km x 30 km centrado em Jundiaí (lon,lat) ---
center = ee.Geometry.Point([-46.8842, -23.1857])   # Jundiaí, SP
aoi = center.buffer(15000).bounds()                # ~30 km de largura

# --- período de teste ---
start = "2020-01-01"
end   = "2020-04-01"  # 1 trimestre

# --- coleção Landsat 8 L2 ---
ic = (ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
      .filterBounds(aoi)
      .filterDate(start, end))

def mask_clouds(img):
    qa = img.select("QA_PIXEL")
    clear = qa.bitwiseAnd(1 << 6).neq(0)
    not_cloud = qa.bitwiseAnd(1 << 3).eq(0)
    not_shadow = qa.bitwiseAnd(1 << 4).eq(0)
    not_dilated = qa.bitwiseAnd(1 << 1).eq(0)
    not_cirrus = qa.bitwiseAnd(1 << 2).eq(0)
    mask = clear.And(not_cloud).And(not_shadow).And(not_dilated).And(not_cirrus)

    # remove saturados
    sat = img.select("QA_RADSAT").eq(0)
    return img.updateMask(mask.And(sat))

def add_ndvi_temp(img):
    # aplicar escala/offset SR e ST (C2 L2)
    red = img.select("SR_B4").multiply(0.0000275).add(-0.2)
    nir = img.select("SR_B5").multiply(0.0000275).add(-0.2)
    ndvi = nir.subtract(red).divide(nir.add(red)).rename("ndvi")

    tempK = img.select("ST_B10").multiply(0.00341802).add(149.0)
    tempC = tempK.subtract(273.15).rename("tempC")

    return ee.Image.cat([ndvi, tempC]).copyProperties(img, img.propertyNames())

ic2 = ic.map(mask_clouds).map(add_ndvi_temp)

# --- composição trimestral ---
ndvi = ic2.select("ndvi").median().rename("ndvi")
tempC = ic2.select("tempC").median().rename("tempC")
n_obs = ic2.select("ndvi").count().rename("n_obs")

out = ee.Image.cat([ndvi, tempC, n_obs]).clip(aoi).toFloat()  # tipo único p/ export

print("Imagens no período:", ic.size().getInfo())
print("OK: composite bands:", out.bandNames().getInfo())

# --- export (Drive) ---
task = ee.batch.Export.image.toDrive(
    image=out,
    description="jundiai_2020Q1_ndvi_temp",
    folder="gee_exports",
    fileNamePrefix="jundiai_2020Q1_ndvi_temp",
    region=aoi,
    scale=30,
    maxPixels=int(1e11),
)
task.start()
print("Export started. Task id:", task.id)

Imagens no período: 5
OK: composite bands: ['ndvi', 'tempC', 'n_obs']
Export started. Task id: MVY4FDYBAZ5HDGDFOP4DQUUD
